# Chapter 7b — Text Analytics
**MADT6004 · Brew Lab BKK case**

Reviews are a free-text leading indicator. Numbers say everything's fine; reviews tell you when it isn't.

You will:
1. Load the reviews
2. Tokenize Thai + English with PyThaiNLP
3. Get top tokens by sentiment
4. Identify the branch whose negative review share is rising


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

# Thai font registration so plots render Thai glyphs
import os, tempfile, urllib.request
import matplotlib, matplotlib.font_manager as fm
FONT_URL  = "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf"
FONT_PATH = os.path.join(tempfile.gettempdir(), "Sarabun-Regular.ttf")
if not os.path.exists(FONT_PATH):
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)
fm.fontManager.addfont(FONT_PATH)
thai_family = fm.FontProperties(fname=FONT_PATH).get_name()
matplotlib.rcParams['font.family'] = [thai_family, 'DejaVu Sans']

from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])
print("Thai font registered:", thai_family)


## 2. Load and tokenize the reviews

In [ ]:
import re
reviews = pd.read_sql("SELECT * FROM reviews", conn)
reviews["review_date"] = pd.to_datetime(reviews["review_date"])
print(f"Total reviews: {len(reviews)}")
print(reviews[["branch_id","rating","sentiment_label","text"]].head())

THAI_STOPS = set(thai_stopwords())
EN_STOPS = {'the','a','an','at','of','to','in','for','is','was','it','this','that','and','or','but'}

def tokenize(text):
    if not isinstance(text, str): return []
    out = []
    for tok in word_tokenize(text, engine="newmm"):
        tok = tok.strip().lower()
        if not tok or tok in THAI_STOPS or tok in EN_STOPS: continue
        if re.fullmatch(r'[\d\W_]+', tok): continue
        if len(tok) == 1 and not re.match(r'[ก-๙]', tok): continue
        out.append(tok)
    return out

reviews["tokens"] = reviews["text"].apply(tokenize)
print("\nSample tokenization:", reviews["tokens"].iloc[0])


## 3. Top tokens overall

In [ ]:
from collections import Counter
all_tokens = [t for toks in reviews["tokens"] for t in toks]
top = pd.Series(Counter(all_tokens)).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index[::-1], top.values[::-1], color="#0891B2")
ax.set_title("Top 20 review tokens")
plt.tight_layout(); plt.show()


## 4. Top tokens by sentiment
Compare which words dominate positive vs negative reviews.

In [ ]:
def top_for(label, n=15):
    sub = reviews[reviews["sentiment_label"] == label]
    toks = [t for r in sub["tokens"] for t in r]
    return pd.Series(Counter(toks)).sort_values(ascending=False).head(n)

pos = top_for("positive"); neg = top_for("negative")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(pos.index[::-1], pos.values[::-1], color="#0891B2")
axes[0].set_title("Positive — top tokens")
axes[1].barh(neg.index[::-1], neg.values[::-1], color="#DC2626")
axes[1].set_title("Negative — top tokens")
plt.tight_layout(); plt.show()


## 5. Negative-review share by branch over time
A simple but powerful heuristic: which branches are seeing complaint volume rise even if average rating stays flat?

In [ ]:
br = pd.read_sql("SELECT branch_id, name FROM branches", conn)
r = reviews.merge(br, on="branch_id")
r["week"] = r["review_date"].dt.to_period("W").dt.start_time
r["is_neg"] = (r["sentiment_label"] == "negative").astype(int)

weekly = (r.groupby(["name","week"])
            .agg(neg_share=("is_neg","mean"), n=("is_neg","size"))
            .reset_index())

# Top 6 branches by review volume — for readability
top_branches = r["name"].value_counts().head(6).index
fig, ax = plt.subplots(figsize=(11, 5))
for name in top_branches:
    sub = weekly[weekly["name"] == name]
    ax.plot(sub["week"], sub["neg_share"], marker="o", label=name, lw=1)
ax.set_ylabel("Negative review share (per week)")
ax.set_title("Weekly negative-review share by branch")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


## Discussion prompts
1. Tokenization decisions (stopwords, length, casing) materially change the picture. What rule did you find most impactful?
2. The negative-share chart can hide branches with low volume. How would you handle a branch with only 5 reviews/week?
3. If you could attach **one** structured feature from the rest of the database to each review, which would you pick?
